In [11]:
from pydantic import BaseModel, Field
from typing import Optional,Literal
import mappings
class LaptopSpecs(BaseModel):
    # ==========================
    # 1. Non-Negotiable Hardware Specs
    # ==========================
    RAM: int = Field(..., ge=4, le=128, description="RAM in GB")
    processor_brand: str = Field(..., description="CPU brand, e.g., 'Intel', 'AMD'")
    dedicated_gpu: int = Field(..., ge=0, le=1, description="1 if dedicated GPU exists, else 0")
    
    # ==========================
    # 2. Brand (Now Optional)
    # ==========================
    brand: Optional[str] = Field(default=None, description="e.g., 'ASUS', 'HP', 'Lenovo'. Omit for market baseline.")

    # ==========================
    # 3. Secondary Specs & Defaults
    # ==========================
    gpu_model: Optional[str] = Field(default=None)
    cpu_model: Optional[str] = Field(default=None)
    display_size: Optional[float] = Field(default=15.6, ge=0.0)
    ddr: Optional[int] = Field(default=4)
    
    # Storage flags
    storage_type: Literal["SSD", "HDD", "BOTH"] = "SSD"
    
    # OS flags
    os: Optional[str] = Field(default = 'windows')
average_cpu_power = mappings.get_average(mappings.cpu_scores)
average_gpu_power = mappings.get_average(mappings.gpu_scores)
def prepare_data(data: dict, pipe):
    # Basic Pydantic validation
    item = RawLaptopInput(**data).model_dump()

    # Plain dictionary assignments
    item['cpu_power'] = mappings.cpu_scores.get(item['cpu_model'],average_cpu_power)
    item['gpu_scores'] = mappings.gpu_scores.get(item['gpu_name'], average_gpu_power)
    item['is_ssd'] = 1 if item['storage_type'] in ['SSD', 'BOTH'] else 0
    item['is_hdd'] = 1 if item['storage_type'] in ['HDD', 'BOTH'] else 0
    
def prepare_data(data: dict, pipe):
    try:
        validated_data = LaptopSpecs(**data)
    except ValidationError as err:
        raise ValueError(f"Invalid input: {err}")

    input_df = pd.DataFrame([validated_data.model_dump()])
    ready_df = input_df[pipe.feature_names_in_]
    return ready_df
def test_data(data: dict, pipe) -> float:
    try:
        data = prepare_data(data, pipe)
    except Exception as e:
        raise e

    prediction = pipe.predict(data)
    return float(prediction[0])